# L1 — Weighted Max-Cut: the problem that is already a QUBO

**2026 Quantum Accelerator · Day 1 · QUBO ladder · step 1 of 5 · ~15 minutes**

**New on this step:** the objective is *already* quadratic. Q is read straight off the graph;
nothing is encoded, nothing is penalised. This is the only step of the ladder where that is true.

**The problem.** Split the nodes of a graph into two groups so that the total weight of the
edges *between* the groups is as large as possible. **Why it matters.** It is the QUBO in its
purest form and the benchmark every quantum-optimisation paper uses. **Plan.** The slide
instance by hand, then a graph too big for a slide.

---
## 1. The slide instance

In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import helpers_ladder as h

labels, W = h.maxcut_slide()          # triangle A, B, C with weights 8, -2, -3
G3 = nx.from_numpy_array(W)
G3 = nx.relabel_nodes(G3, dict(enumerate(labels)))
pos = {"A": (0, 1), "B": (-1, 0), "C": (1, 0)}
nx.draw(G3, pos, with_labels=True, node_color="#dbe9f6", node_size=900, font_size=14)
nx.draw_networkx_edge_labels(G3, pos, edge_labels={(a, b): d["weight"] for a, b, d in G3.edges(data=True)})
plt.title("w_AB = 8, w_AC = -2, w_BC = -3")
plt.show()

---
## 2. Q by hand

In [2]:
W = np.array([[0, 8, -2],             # <-- CHANGE ME: flip a sign, re-run sections 2-4
              [8, 0, -3],
              [-2, -3, 0]])
Q = -np.diag(W.sum(axis=1)) + W          # Q_ii = -(sum of the weights at node i),  Q_ij = +w_ij
h.print_matrix(Q, ["A", "B", "C"], "Q (minimise x^T Q x = minus the cut weight)")

Q (minimise x^T Q x = minus the cut weight)
       A   B   C
   A  -6   8  -2
   B   8  -5  -3
   C  -2  -3   5


**What you see.** The slide minimises `-cut`, so every number is a negative cut weight. Why is
`Q_ii` the negative row sum? An edge is cut when its two ends differ, and
`x_i + x_j - 2 x_i x_j` is exactly 1 in that case and 0 otherwise (slide 43). Summing over
edges puts `+w_ij` on each pair and `-w_ij` on both diagonal entries it touches.

---
## 3. Every answer

In [3]:
table = h.exact_table(Q, labels, ties_asc=False)
table["cut"] = -table["energy"]
table

,A,B,C,energy,objective,cut
0,1,0,0,-6,-6,6
1,0,1,1,-6,-6,6
2,1,0,1,-5,-5,5
3,0,1,0,-5,-5,5
4,1,1,1,0,0,0
5,0,0,0,0,0,0
6,1,1,0,5,5,-5
7,0,0,1,5,5,-5


**What you see.** The two rows at −6 are the *same* cut: A alone against B and C together,
and its mirror image. Swapping the two groups never changes the cut, so every Max-Cut QUBO has
this **global flip symmetry**. The first row is the slide's "Cut Cost = 6".

---
## 4. The same Q from PyQUBO

In [4]:
from pyqubo import Binary

x = {k: Binary(k) for k in labels}
cut = sum(W[i, j] * (x[a] + x[b] - 2 * x[a] * x[b])
          for i, a in enumerate(labels) for j, b in enumerate(labels) if i < j)
model = (-cut).compile()
qubo, offset = model.to_qubo()
Q_lib = h.symmetric_from_qubo_dict(qubo, labels)
print(np.allclose(Q, Q_lib), offset)      # True 0.0

True 0.0


**What you see.** `True`: the library built the same matrix we wrote by hand, and there is no
constant. (PyQUBO returns an upper-triangular dictionary; the helper halves and mirrors the
off-diagonals to compare it with the slide's symmetric Q.)

---
## 5. Now too big for a slide

In [5]:
import time

W_big, G = h.maxcut_instance(n=120, p=0.15, seed=1)       # <-- CHANGE ME (stretch): n=120
Q_big = h.maxcut_Q(W_big)
ss, dt = h.anneal(Q_big)                                   # 100 reads x 2000 sweeps, seed 1

R = h.read_matrix(ss, 120)
cuts = np.array([h.cut_value(W_big, x) for x in R])
best_x, _ = h.best_row(ss, 120)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges, total edge weight {W_big.sum()/2:.0f}")
print(f"SA: best cut {cuts.max():.0f}, mean {cuts.mean():.0f}, worst {cuts.min():.0f}   ({dt:.2f} s)")

120 nodes, 1072 edges, total edge weight 5891
SA: best cut 4061, mean 4052, worst 4003   (0.35 s)


The honest reference: an exact solver (CP-SAT) with a 30-second budget, and random assignments as a floor.

In [6]:
ref = h.cpsat_maxcut(W_big, time_limit=30)                # <-- CHANGE ME: 30 -> 60
print(f"CP-SAT: value {ref['value']:.0f}, bound {ref['bound']:.0f}, status {ref['status']}, {ref['seconds']:.2f} s")

rng = np.random.default_rng(0)
random_mean = np.mean([h.cut_value(W_big, x) for x in rng.integers(0, 2, (100, 120))])
print(f"random assignments: mean cut {random_mean:.0f}")

plt.hist(cuts, bins=15, color="#1f77b4", alpha=0.8)
plt.axvline(ref["value"], color="crimson", ls="--", label=f"CP-SAT, 30 s: {ref['value']:.0f}")
plt.axvline(random_mean, color="grey", ls=":", label=f"random mean: {random_mean:.0f}")
plt.xlabel("cut weight of one SA read"); plt.ylabel("reads"); plt.legend(); plt.show()
h.draw_cut(G, W_big, best_x)

CP-SAT: value 3957, bound 5891, status FEASIBLE, 30.00 s
random assignments: mean cut 2957


**What you see.** The annealer beats the exact solver by about 2.6 % in a hundredth of the
time, and the exact solver cannot even say how far from optimal it is: its bound equals the
total edge weight, so the gap is about 49 %. This is the regime notebook 05 is built on. The
point of this ladder step is only that *no encoding happened* — each of the next four steps adds one.

---
## 6. What changed on this step

- Q came straight from the graph: `Q_ii = -(row sum)`, `Q_ij = +w_ij`. No penalty, no parameter.
- Two optimal rows are one cut: every Max-Cut QUBO has a global flip symmetry.
- Hand-built Q and PyQUBO agree exactly, with offset 0.
- At n = 120 the annealer is cheap and competitive; the exact solver's bound never moved.
- Next, **L2**: the objective is a *square*, and the diagonal appears from `x² = x`.

---
## Stretch

**A.** Set all three slide weights positive. Is the optimum still a "one versus two" split?

**B.** Use `n=250`. How does SA time scale, and does CP-SAT's bound move at all?

**C.** Make the big graph bipartite (`nx.complete_bipartite_graph`). The optimum is then the
total weight — does SA find it, and does CP-SAT prove it?